# Phase 6 : Évaluation du modèle

**Objectif :** Mesurer la qualité des recommandations avec des métriques objectives.

## Métriques :
- **Precision** : % de recommandations pertinentes
- **Recall** : % de films appréciés retrouvés
- **F1-Score** : Moyenne harmonique Precision/Recall
- **RMSE** : Erreur de prédiction des notes

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, precision_score, recall_score
import sys, os
sys.path.append(os.path.abspath('..'))
from backend.recommender import MovieRecommender
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Chargement du modèle et des données
recommender = MovieRecommender.load('../models/recommendation_model.pkl')
train_data = pd.read_csv('../data/train_data.csv')
test_data = pd.read_csv('../data/test_data.csv')

print('✅ Modèle et données chargés')

In [ ]:
# 1. PRECISION & RECALL
def evaluate_precision_recall(recommender, test_data, threshold=3.5, n_recs=10):
    """
    Calcule Precision et Recall.
    
    Precision = Vrais positifs / (Vrais positifs + Faux positifs)
    Recall = Vrais positifs / (Vrais positifs + Faux négatifs)
    """
    test_data['liked'] = (test_data['rating'] >= threshold).astype(int)
    precisions, recalls = [], []
    
    for user_id in test_data['userId'].unique():
        user_test = test_data[test_data['userId'] == user_id]
        true_liked = set(user_test[user_test['liked'] == 1]['movieId'])
        
        # Films recommandés
        recommended = recommender.recommend(user_id, n_recommendations=n_recs)
        recommended_ids = set(m['movieId'] for m in recommended)
        
        # Vrais positifs
        true_pos = len(true_liked & recommended_ids)
        
        if len(recommended_ids) > 0:
            precisions.append(true_pos / len(recommended_ids))
        if len(true_liked) > 0:
            recalls.append(true_pos / len(true_liked))
    
    precision = np.mean(precisions) if precisions else 0
    recall = np.mean(recalls) if recalls else 0
    f1 = 2 * (precision * recall) / (precision + recall + 1e-10)
    
    return {'precision': precision, 'recall': recall, 'f1_score': f1}

metrics = evaluate_precision_recall(recommender, test_data)

print('📊 MÉTRIQUES DE RECOMMANDATION')
print('='*40)
print(f'Precision : {metrics["precision"]:.2%}')
print(f'Recall    : {metrics["recall"]:.2%}')
print(f'F1-Score  : {metrics["f1_score"]:.2%}')
print(f'\n👉 Interprétation :')
print(f'   - {metrics["precision"]:.0%} des films recommandés sont pertinents')
print(f'   - {metrics["recall"]:.0%} des films appréciés ont été recommandés')

In [ ]:
# 2. RMSE (Root Mean Square Error)
def calculate_rmse(recommender, test_data):
    """Calcule l'erreur quadratique moyenne des prédictions"""
    predictions, actuals = [], []
    
    for _, row in test_data.sample(min(1000, len(test_data))).iterrows():
        recommended = recommender.recommend(row['userId'], n_recommendations=50)
        predicted = next(
            (m['predicted_rating'] for m in recommended if m['movieId'] == row['movieId']),
            None
        )
        if predicted:
            predictions.append(predicted)
            actuals.append(row['rating'])
    
    return np.sqrt(mean_squared_error(actuals, predictions))

rmse = calculate_rmse(recommender, test_data)
print(f'📉 RMSE : {rmse:.3f}')
print(f'   (Erreur moyenne de prédiction : {rmse:.2f} point sur 5)')
print(f'   👉 Plus le RMSE est bas, mieux c\'est (idéalement < 1)')

In [ ]:
# 3. ANALYSE PAR SEUIL
thresholds = [3.0, 3.5, 4.0, 4.5]
print('📊 IMPACT DU SEUIL DE PERTINENCE')
print('='*50)
print(f'{"Seuil":<10} {"Precision":<12} {"Recall":<12} {"F1-Score":<12}')
print('-'*50)

for t in thresholds:
    m = evaluate_precision_recall(recommender, test_data, threshold=t)
    print(f'{t:<10} {m["precision"]:<12.2%} {m["recall"]:<12.2%} {m["f1_score"]:<12.2%}')

In [ ]:
# 4. SYNTHÈSE
print('='*60)
print('RÉSUMÉ DE L\'ÉVALUATION')
print('='*60)
print(f'''
✅ Le modèle de filtrage collaboratif a été entraîné avec succès.
📊 Precision : {metrics["precision"]:.1%} des recommandations sont pertinentes.
📊 Recall : {metrics["recall"]:.1%} des bons films sont retrouvés.
📊 F1-Score : {metrics["f1_score"]:.1%} (bon équilibre).
📉 RMSE : {rmse:.3f} (faible erreur de prédiction).

🔮 Améliorations possibles :
   - Utiliser la factorisation matricielle (SVD)
   - Ajouter un filtrage basé contenu (genres)
   - Augmenter la taille du dataset
   - Ajuster le nombre de voisins (K)
''')